# Практика 8. Публикуем модель из Colab

К лекции «Интерпретируемость, надёжность и продакшен».
Короткий маршрут: обучить один Pipeline, объяснить признаки, сохранить модель,
дать другому человеку форму по временной ссылке.
Подбор моделей и гиперпараметров уже изучен; здесь его не повторяем.

1. Обучаем модель и коротко проверяем качество и важность признаков.
2. Сохраняем пайплайн и запускаем приложение, которое читает файлы с диска.
3. Получаем публичную ссылку Gradio и проверяем форму с другого устройства.
4. Останавливаем сервер и проверяем, что новые запросы больше не выполняются.

Все ячейки готовы к запуску сверху вниз. GPU, платная подписка и токены не нужны.
Отдельная пошаговая инструкция: [инструкция со скриншотами](practice08_publishing_guide.md).

Модель работает в среде Colab. Посетитель открывает отдельную страницу с формой;
доступ к блокноту ему не нужен. После остановки сервера или отключения среды
ссылка перестаёт обслуживать запросы. Это учебная временная публикация.

[Открыть в Google Colab](https://colab.research.google.com/github/rustam-azimov/ml-course-hse/blob/main/practice/practice08_publish_model.ipynb)

In [ ]:
%pip install -q gradio==6.0.1 shap==0.51.0

## 1. Обучаем один пайплайн

Берём четыре исходных признака. Внутри Pipeline живут заполнение пропусков,
кодирование категорий и классификатор. Приложению не придётся повторять предобработку.
Разделение стратифицированное; тест используем для итоговой оценки.
Порог 0.5 выбран заранее для демонстрации и сохраняется вместе с моделью.

In [ ]:
import hashlib
import json
import platform
from pathlib import Path
from urllib.request import urlopen

import joblib
import numpy as np
import pandas as pd
import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

DATA_URL = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/titanic.csv"
data_bytes = urlopen(DATA_URL, timeout=30).read()
from io import BytesIO

df = pd.read_csv(BytesIO(data_bytes))
FEATURES = ["age", "fare", "sex", "pclass"]
X_train, X_test, y_train, y_test = train_test_split(
    df[FEATURES], df["survived"], test_size=0.2,
    stratify=df["survived"], random_state=42,
)
prep = ColumnTransformer([
    ("numeric", SimpleImputer(strategy="median"), ["age", "fare", "pclass"]),
    ("category", OneHotEncoder(handle_unknown="ignore", sparse_output=False), ["sex"]),
])
model = Pipeline([
    ("prep", prep),
    ("classifier", RandomForestClassifier(
        n_estimators=100, min_samples_leaf=5, random_state=42, n_jobs=1,
    )),
])
model.fit(X_train, y_train)
test_proba = model.predict_proba(X_test)[:, 1]
auc = roc_auc_score(y_test, test_proba)
THRESHOLD = 0.5
print(f"Обучение: {len(X_train)} объектов; тест: {len(X_test)}")
print(f"ROC-AUC: {auc:.3f}; константный прогноз: 0.500")
assert auc > 0.7, "Проверьте данные и признаки"

## 2. Что использует модель

Permutation importance измеряет падение ROC-AUC после перемешивания одного столбца.
Отрицательный результат возможен из-за шума оценки. Важность не доказывает причинность.
Этот разбор не используется для изменения модели после просмотра теста.

In [ ]:
importance = permutation_importance(
    model, X_test, y_test, scoring="roc_auc", n_repeats=5, random_state=42,
)
display(pd.DataFrame({
    "признак": FEATURES,
    "падение ROC-AUC": importance.importances_mean,
}).sort_values("падение ROC-AUC", ascending=False))

### 2.1. SHAP: объясняем один прогноз

Запустите ячейку после обучения модели. Берём пассажирку 30 лет,
билет стоимостью 8, третий класс. SHAP покажет, какие признаки повышают
или понижают её прогноз относительно среднего на 100 обучающих примерах.

Красный повышает оценку, синий понижает. На графике значения переведены
в проценты: вклад 22.7 означает +22.7 процентного пункта. Нижнее E[f(X)] —
средний прогноз, верхнее f(x) — прогноз пассажира. Ожидаем около 38.6% и 43.3%.
Это объяснение модели, а не доказательство причинности.

In [ ]:
import shap
import matplotlib.pyplot as plt

prep = model.named_steps["prep"]
background = shap.sample(prep.transform(X_train), 100, random_state=42)
explainer = shap.TreeExplainer(
    model.named_steps["classifier"], data=background,
    feature_perturbation="interventional", model_output="probability",
)
row = pd.DataFrame([dict(age=30, fare=8, sex="female", pclass=3)])
sv = explainer(prep.transform(row))[0, :, 1]  # 1 — класс «выжил»

# Возраст, стоимость, класс; два one-hot столбца пола складываем.
values = np.r_[sv.values[:3], sv.values[3:].sum()]
np.testing.assert_allclose(
    sv.base_values + values.sum(), model.predict_proba(row)[0, 1], atol=1e-7,
)
explanation = shap.Explanation(
    values=values * 100, base_values=float(sv.base_values) * 100,
    data=[30, 8, 3, "женский"],
    feature_names=["Возраст", "Стоимость билета", "Класс билета", "Пол"],
)
shap.plots.waterfall(explanation, show=False)
plt.xlabel("Оценка вероятности выживания, %")
plt.tight_layout()
plt.show()
print(f"Средний прогноз: {sv.base_values:.1%}")
print(f"Прогноз пассажира: {model.predict_proba(row)[0, 1]:.1%}")

## 3. Сохраняем и восстанавливаем

Сохраняем весь пайплайн, порог, качество, источник и хэш данных, версии библиотек.
Загружайте через joblib только собственные или доверенные файлы: загрузка может исполнять код.
Каталог `titanic_demo` будет содержать всё необходимое для запуска вне ноутбука.

In [ ]:
import gradio as gr

EXPORT = Path("titanic_demo")
EXPORT.mkdir(exist_ok=True)
joblib.dump(model, EXPORT / "model.joblib")
metadata = {
    "version": "1.0.0", "threshold": THRESHOLD,
    "test_roc_auc": float(auc), "features": FEATURES,
    "data_url": DATA_URL, "data_sha256": hashlib.sha256(data_bytes).hexdigest(),
    "random_state": 42,
    "versions": {"scikit-learn": sklearn.__version__, "pandas": pd.__version__,
                 "numpy": np.__version__, "joblib": joblib.__version__,
                 "gradio": gr.__version__, "python": platform.python_version()},
}
(EXPORT / "metadata.json").write_text(
    json.dumps(metadata, ensure_ascii=False, indent=2), encoding="utf-8",
)
loaded = joblib.load(EXPORT / "model.joblib")
np.testing.assert_allclose(loaded.predict_proba(X_test)[:, 1], test_proba)
print("Прогноз после загрузки совпадает с исходным ✓")

## 4. Создаём отдельное приложение

`app.py` загружает файлы с диска. Оно не обращается к переменным ноутбука и не обучает модель
при запуске. Функция `predict` проверяет вход, а сохранённый Pipeline обрабатывает признаки.
Обучение и сервис используют одну предобработку.

In [ ]:
APP_CODE = '''
import json
from pathlib import Path

import gradio as gr
import joblib
import numpy as np
import pandas as pd

ROOT = Path(__file__).resolve().parent
model = joblib.load(ROOT / "model.joblib")
meta = json.loads((ROOT / "metadata.json").read_text(encoding="utf-8"))


def predict(age, fare, sex, pclass):
    # Пустой возраст допустим: медиану подставляет сохранённый Pipeline.
    age = np.nan if age is None else float(age)
    if not np.isnan(age) and (not np.isfinite(age) or not 0 <= age <= 120):
        raise gr.Error("Возраст должен быть от 0 до 120 или пустым.")
    if fare is None or not np.isfinite(fare) or not 0 <= fare <= 10000:
        raise gr.Error("Стоимость билета должна быть от 0 до 10000.")
    if sex not in ("female", "male") or pclass not in (1, 2, 3):
        raise gr.Error("Выберите пол и класс билета из списка.")
    row = pd.DataFrame([dict(age=age, fare=fare, sex=sex, pclass=pclass)])
    probability = float(model.predict_proba(row)[0, 1])
    return {
        "probability": round(probability, 4),
        "prediction": int(probability >= meta["threshold"]),
        "threshold": meta["threshold"],
        "model_version": meta["version"],
    }


demo = gr.Interface(
    fn=predict,
    inputs=[
        gr.Number(value=30, label="Возраст (можно оставить пустым)"),
        gr.Number(value=100, label="Стоимость билета"),
        gr.Dropdown(["female", "male"], value="female", label="Пол"),
        gr.Dropdown([1, 2, 3], value=1, label="Класс билета"),
    ],
    outputs=gr.JSON(label="Прогноз"),
    examples=[[30, 100, "female", 1], [40, 8, "male", 3]],
    title="Титаник: учебная модель",
    description="Исторические данные. Не применять для реальных решений. "
                "probability — оценка модели; prediction — класс по сохранённому порогу.",
)

if __name__ == "__main__":
    demo.launch()
'''
(EXPORT / "app.py").write_text(APP_CODE, encoding="utf-8")
model_card = f"""# Карточка модели

Учебная бинарная классификация на исторических данных Titanic (seaborn-data).
Версия: {metadata['version']}. Признаки: {', '.join(FEATURES)}.
Обучение: {len(X_train)} объектов, тест: {len(X_test)}; seed=42.
ROC-AUC на тесте: {auc:.3f}. Порог: {THRESHOLD}, выбран заранее.
Источник, SHA-256 данных и версии окружения: metadata.json.

Модель оценивает выживание пассажира в данных одного события 1912 года.
Она не предназначена для современных задач и реальных решений о людях.
Вероятности не калиброваны. Пол и класс билета отражают исторические различия;
важность признаков не доказывает причинность.

Перед реальной эксплуатацией нужны мониторинг ошибок и задержки, пропусков,
распределений признаков и качества по новой разметке, а также возможность отката.
Демонстрация этого не реализует. Приложение работает только пока запущена среда Colab.
"""
(EXPORT / "README.md").write_text(model_card, encoding="utf-8")
print("Файлы приложения:", sorted(p.name for p in EXPORT.iterdir()))

## 5. Проверяем приложение до публикации

Импортируем записанный файл и вызываем его функцию. Проверяем обычный запрос,
пропущенный возраст, отклонение неверных значений и совпадение с прогнозом из ноутбука.
Новая категория не ломает внутренний кодировщик, но форма принимает только допустимый пол.

In [ ]:
import importlib.util

spec = importlib.util.spec_from_file_location("titanic_app", EXPORT / "app.py")
app = importlib.util.module_from_spec(spec)
spec.loader.exec_module(app)

regular = app.predict(30, 100, "female", 1)
missing = app.predict(None, 8, "male", 3)
row = pd.DataFrame([dict(age=30, fare=100, sex="female", pclass=1)])
assert regular["probability"] == round(float(model.predict_proba(row)[0, 1]), 4)
for result in (regular, missing):
    assert 0 <= result["probability"] <= 1
    assert result["prediction"] == int(result["probability"] >= result["threshold"])
for args in [(-1, 100, "female", 1), (30, -5, "female", 1),
             (30, 100, "unknown", 1), (30, 100, "female", 4)]:
    try:
        app.predict(*args)
    except gr.Error:
        pass
    else:
        raise AssertionError(f"Некорректный запрос принят: {args}")
unknown = row.assign(sex="unknown")
assert np.isfinite(loaded.predict_proba(unknown)).all()
print("Обычный запрос:", regular)
print("Без возраста:", missing)
print("Совпадение прогнозов и проверка входа ✓")

## 6. Публикуем временную ссылку из Colab

Следующая ячейка создаст общедоступную страницу с формой. Скопируйте URL `https://….gradio.live`
из вывода и откройте в другом браузере или на телефоне. Вводите только учебные данные.
Посетителю не нужен доступ к блокноту: модель считает на вашей машине Colab.
После остановки сервера или отключения среды Colab ссылка перестанет работать;
у самой ссылки тоже ограниченный срок жизни. Это временная демонстрация.

Если повторно запускаете ячейку, сначала выполните `app.demo.close()`.

In [ ]:
app.demo.launch(share=True, debug=False, prevent_thread_lock=True)

## 7. Проверяем ссылку и завершаем демонстрацию

1. Откройте публичную ссылку на телефоне или в другом браузере.
2. Введите `30, 100, female, 1` и сравните ответ с `regular` выше.
3. Очистите возраст: прогноз должен появиться. Возраст `-1` должен вызвать ошибку.
4. Дайте одногруппнику выполнить новый запрос, пока ваша среда Colab подключена.
5. После проверки создайте ячейку с `app.demo.close()` и выполните её.
   Повторите запрос по публичной ссылке: нового прогноза быть не должно.

Команду остановки запускаем вручную после демонстрации. Уже выведенный в браузере
ответ может остаться на экране — это не означает, что сервер работает.
Для повторного запуска выполните ячейку раздела 6 и используйте адрес из её вывода.
После перезапуска среды Colab потребуется заново выполнить все ячейки.

Ссылка общедоступна и имеет ограниченный срок жизни. Colab может отключить среду
раньше. Не используйте этот вариант для сервиса с гарантированной доступностью.
[Документация Gradio](https://www.gradio.app/guides/sharing-your-app).

### Вопрос

Где выполняется модель, когда одногруппник открывает ссылку? Что произойдёт после остановки сервера? Какие показатели нужно мониторить у работающего сервиса?

_Ваш ответ:_

